# Step 2 — Plant→Pollinator Edge List
From the GloBI dump, filter pollination interactions for the top 50 plant species (rank 2–51 from PhenoField train set), normalize directionality, and output unique (plant_species, pollinator_species) pairs.

## 0. Download GloBI Data
Run this once in terminal to download the GloBI interactions snapshot:
```bash
wget -O interactions.csv.gz "https://zenodo.org/records/20546682/files/interactions.csv.gz"
```

In [ ]:
import pandas as pd

## 1. Plant Species List (Rank 2–51, excluding 'unknown')

In [ ]:
plant_species = [
    "Asimina triloba", "Sanguinaria canadensis", "Mitchella repens",
    "Cypripedium acaule", "Erodium cicutarium", "Malosma laurina",
    "Diospyros virginiana", "Dipterostemon capitatus", "Phytolacca americana",
    "Trillium grandiflorum", "Claytonia virginica", "Microstegium vimineum",
    "Trillium ovatum", "Lonicera maackii", "Asclepias syriaca",
    "Rhus glabra", "Ligustrum sinense", "Aquilegia canadensis",
    "Passiflora incarnata", "Chimaphila maculata", "Trillium erectum",
    "Alliaria petiolata", "Celastrus orbiculatus", "Amphicarpaea bracteata",
    "Arisaema triphyllum", "Bignonia capreolata", "Houstonia caerulea",
    "Dicentra cucullaria", "Impatiens capensis", "Conium maculatum",
    "Mertensia virginica", "Convolvulus arvensis", "Glechoma hederacea",
    "Erythronium americanum", "Galium aparine", "Achillea millefolium",
    "Lysimachia borealis", "Caltha palustris", "Sambucus canadensis",
    "Staphylea trifolia", "Kalmia latifolia", "Maianthemum racemosum",
    "Lamium purpureum", "Triteleia laxa", "Erigeron philadelphicus",
    "Chamaenerion angustifolium", "Bellis perennis", "Eschscholzia californica",
    "Sambucus cerulea", "Larrea tridentata"
]

## 2. Filter and Build Edge List

In [ ]:
interaction_types = {
    'pollinates', 'visits', 'visitsFlowersOf',
    'visitedBy', 'flowersVisitedBy', 'hasFlowerVisitor', 'pollinatedBy'
}

# *By types have source/target inverted — pollinator is source, plant is target
inverted_types = {'visitedBy', 'flowersVisitedBy', 'hasFlowerVisitor', 'pollinatedBy'}

cols_needed = ['sourceTaxonSpeciesName', 'targetTaxonSpeciesName', 'interactionTypeName']

chunks = []
for i, chunk in enumerate(pd.read_csv('interactions.csv.gz', sep=',', usecols=cols_needed,
                                       chunksize=100_000, low_memory=False)):
    if i % 10 == 0:
        print(f"Processing chunk {i}...")

    # Keep only pollination interactions
    chunk = chunk[chunk['interactionTypeName'].isin(interaction_types)].copy()

    # Normalize directionality: swap *By types so plant is always source
    mask = chunk['interactionTypeName'].isin(inverted_types)
    chunk.loc[mask, ['sourceTaxonSpeciesName', 'targetTaxonSpeciesName']] = \
        chunk.loc[mask, ['targetTaxonSpeciesName', 'sourceTaxonSpeciesName']].values

    # Filter to target plant species
    chunk = chunk[chunk['sourceTaxonSpeciesName'].isin(plant_species)]
    chunks.append(chunk)

df = pd.concat(chunks)
df = df.rename(columns={
    'sourceTaxonSpeciesName': 'plant_species',
    'targetTaxonSpeciesName': 'pollinator_species'
})

edge_list = df[['plant_species', 'pollinator_species']].drop_duplicates().dropna()
print(f"\n{len(edge_list)} unique (plant, pollinator) pairs")

## 3. Inspect Results

In [ ]:
print(edge_list['plant_species'].value_counts())

## 4. Save Edge List

In [ ]:
edge_list.to_csv('plant_pollinator_edges.csv', index=False)
print("Saved to plant_pollinator_edges.csv")